# Timed acquisitions

```{admonition} How To Work
:class: important
The **Preparatory Homework (BAS)** part of this manual should be completed **individually**.

The **Practicum (IC)** part should be completed **in pairs**. During the practicum, divide the workload as equally as possible. For example, while one student sets up the ALPACA connections, the other can prepare the code. Then swap roles for the next measurement.

Throughout this manual, keep your calculations, predictions, and observations in your log book entry. The comparison tables from Compare and Conclude should be completed in your log book entry and included in your onepager report for this lab.

Download the entry from the download menu at the top of this page, under **Log book**. Complete the **Preparation** side before the practicum, then fill in the **Practicum** side at the bench. The PDF has interactive fields, so you can either type into it or print it and write by hand. If you would rather draw the sketches or add a photo than print the entry, open it in [pdfannotation.app](https://pdfannotation.app/), which handles form filling, drawing and images together.
```

```{typst} 2.2_Timed_acquisitions_logbook.typ
:label: Entry 2.2
:fillable:
```


## Preparatory Homework

### Background

```{admonition} Preparation
:class: tip
Before starting this experiment, make sure that you are familiar with the following introductory manuals:

* [ALPACA 2.0 and its onboard voltmeter](../intros/alpaca2/_intro.ipynb)
* [Programming the Student Pico](../intros/alpaca2/programming.ipynb)
* [ALPACA ADC inputs and signal generation](../intros/alpaca2/signals.ipynb)
* [Helper function generator](../intros/alpaca2/funcgen.ipynb)
* [The Playground and its variable voltage](../intros/alpaca2/sandbox_playground.ipynb#variable-u)
* [Oscilloscope](../intros/oscilloscope.ipynb)

Revisit lecture slides and your notes if needed on:

* Time measurement and timing precision
* Data sampling and sampling rate
* Nyquist theorem (basic concepts)
```

When measuring signals with the ALPACA, accurate timing is fundamental. The following concepts are important for understanding timed acquisitions:

**Timing Scales in MicroPython**: The ALPACA runs MicroPython, which provides different timing functions. `time.ticks_ms()` measures time in milliseconds (ms), while `time.ticks_us()` measures in microseconds (μs). Choosing the right scale depends on your measurement duration and required precision. Use `time.ticks_add()` and `time.ticks_diff()` when scheduling or comparing ticks, because the clock can roll over.

**Sampling Rate**: The number of measurements per second determines whether you can capture signal details accurately. The faster you sample, the more points you capture in each signal cycle, but you also collect more data and use more memory.

**Time Recording**: Storing the time of each measurement alongside the voltage value allows you to reconstruct signal behavior accurately and calculate rates of change.

### Anticipate

(Task_A1_2_2)=
#### Task A1: Timing precision

```{admonition} Estimated time: 10 min
:class: estimated-time no-content
```

The ALPACA has two timing functions: `time.ticks_ms()` (milliseconds) and `time.ticks_us()` (microseconds).

1. If you want to measure a signal for 5 seconds with a sampling interval of 10 ms, how many samples will you collect?
2. Which timing function would you choose for this measurement, and why?
3. If reading and storing each value takes about $2\,\mathrm{ms}$ and your loop then sleeps for $10\,\mathrm{ms}$, estimate the interval between readings.
4. For 100 readings with a target interval of $50\,\mathrm{ms}$: how many intervals are there between the first and last readings? Assuming each interval is $50\,\mathrm{ms}$, predict the total time from the first reading to the last.

Write your reasoning in your notepad.

(Task_A2_2_2)=
#### Task A2: Sampling rate and signal representation

```{admonition} Estimated time: 15 min
:class: estimated-time no-content
```

You want to measure a sine wave with frequency $f = 100\,\mathrm{Hz}$.

1. Calculate the period of this sine wave.
2. To accurately capture the signal shape, you generally need at least 10 samples per period. Calculate the minimum sampling rate (samples/second) required.
3. If your loop can execute 100 iterations per second, is this sufficient for your 100 Hz signal?
4. What is the maximum sampling rate you could achieve with the ALPACA if each measurement loop takes approximately 10 ms?

Record your calculations in your notepad.

(Task_A3_2_2)=
#### Task A3: Data storage and measurement strategy

```{admonition} Estimated time: 10 min
:class: estimated-time no-content
```

You are given three measurement strategies:

**Strategy 1**: Record voltage only (no time), and calculate time later assuming perfectly regular intervals.

**Strategy 2**: Record both voltage and time for each measurement.

**Strategy 3**: Record multiple voltage measurements, then record time after they are all collected.

1. What are the advantages and disadvantages of each strategy?
2. Under what circumstances would Strategy 1 fail (i.e., when would your assumption of regular intervals be wrong)?
3. Which strategy would you choose for measuring a 100 Hz sine wave on the ALPACA, and why?

Write your reasoning in your notepad.

(Task_A4_2_2)=
#### Task A4: Frequency sweep planning

```{admonition} Estimated time: 15 min
:class: estimated-time no-content
```

A frequency sweep means measuring the same circuit response at multiple different signal frequencies. This is useful for characterizing circuit behavior across a frequency range.

Consider sweeping a circuit from $50\,\mathrm{Hz}$ to $500\,\mathrm{Hz}$ in steps of $50\,\mathrm{Hz}$. At each frequency, collect 1000 samples at a target interval of $0.2\,\mathrm{ms}$. Wait $0.5\,\mathrm{s}$ for settling before each acquisition.

1. How many different frequencies will you test?
2. For each frequency, how long is the nominal acquisition window (1000 samples × $0.2\,\mathrm{ms}$)? How long is the interval from the first sample to the last?
3. What is the total nominal measurement time, including the $0.5\,\mathrm{s}$ settling time before each acquisition?
4. Is this measurement duration practical for a lab session?

Record your calculations in your notepad.

## Practicum
### Implement and Investigate


During the practicum, work in pairs. One student sets up the ALPACA while the other prepares the acquisition code. Check the connections together and swap roles for the next measurement.

(Task_I1_2_2)=
#### Task I1: Setting up a simple timed measurement

```{admonition} Estimated time: 30 min
:class: estimated-time no-content
```

Use the Student Pico and the onboard `Uvar1` knob. The knob is already powered on the ALPACA. It can produce negative voltages and voltages above the ADC range. Keep it between about $0.5\,\mathrm{V}$ and $2.5\,\mathrm{V}$ for this task.

1. Connect `Uvar1` to `VOLTMETER-IN`. Select `VOLT` on the ALPACA screen. Adjust the knob to about $1.5\,\mathrm{V}$ and leave it still.
2. Set ADC0 to DC coupling, 1x input gain and 1x instrumentation amplifier gain. Leave the external gain resistor connector `J302` open. Set the `J301` instrumentation amplifier reference jumper between `IA REF` and `AGND`.
3. After checking the voltage, connect `Uvar1` to `U+` of **ADC0**. Jumper `U-` to `GNDA` on the ADC0 connector. Keep the voltmeter connected while you measure.
4. In a Student Pico task, read ADC0 with `ADC(26).read_u16()`. Collect 100 readings and their timestamps. Aim for $50\,\mathrm{ms}$ between readings.
5. Convert the counts to volts using the nominal ADC reference of $3.0\,\mathrm{V}$. Use `voltage = count * 3.0 / 65535` for the settings above. The ADC reference is distinct from the `J301` amplifier reference. For a more accurate conversion, temporarily move the `VOLTMETER-IN` lead from `Uvar1` to the `ADC Ref` pin. Use the displayed reference voltage in the formula, then return the lead to `Uvar1`.
6. Return the data to the laptop after acquisition. Print the first and last ten time and voltage pairs. Compare the voltage with the onboard voltmeter reading.

```{tip}
Schedule each reading against a target tick instead of sleeping for a fixed time after each reading. Use `ticks_add()` to find the next target and `ticks_diff()` to compare it with the current tick. Record the actual tick near each ADC read. Do not print inside the acquisition loop.
```

Calculate the first-to-last time, the minimum, maximum and mean interval, and the interval range. Record them in your log book entry. The target is $50\,\mathrm{ms}$; the actual intervals can differ.


(Task_I2_2_2)=
#### Task I2: Measuring a slowly changing signal

```{admonition} Estimated time: 15 min
:class: estimated-time no-content
```

Keep the setup from [Task I1](Task_I1_2_2). Start another acquisition. Slowly turn `Uvar1` while your partner watches the onboard voltmeter. Keep the voltage between about $0.5\,\mathrm{V}$ and $2.5\,\mathrm{V}$.

1. Plot the recorded voltage against the measured time. Label both axes and units.
2. Describe how the voltage changed as you turned the knob.
3. Select a roughly straight part of the plot. Estimate its slope in $\mathrm{V/s}$ and state the two points you used.
4. Did the readings show the change smoothly enough for your slope estimate? Explain using the observed intervals. Which part of the time trace gives the clearest estimate of the slope?

Record the plot and your observations in your log book entry.


(Task_I3_2_2)=
#### Task I3: Generate and inspect a sine wave

```{admonition} Estimated time: 20 min
:class: estimated-time no-content
```

Use the **Helper function generator** as the signal source. You control it from a Student Pico task through the `helper` module. The Helper DAC is separate from Student DAC A and Student DAC B.

1. Disconnect `Uvar1` from `ADC0 U+` and leave ADC0 disconnected from the Helper DAC for now.
2. Place a jumper on `J406` to bypass the Helper DAC output coupling capacitor. Connect the oscilloscope probe tip to `DAC` on the Helper DAC connector `J402`. Connect the probe ground clip to `OSCILLOSCOPE GROUND`. Use DC coupling on the oscilloscope.
3. Write a Student Pico task that starts a $50\,\mathrm{Hz}$ sine with `helper.function_generator.sine(50, 1.0)` and keeps it running for at least $30\,\mathrm{s}$. The second argument requests a $1.0\,\mathrm{V}$ peak-to-peak waveform at the Helper's internal ladder. The voltage at the connector can differ because it passes through an analogue output circuit.
4. Use the oscilloscope to measure the frequency, minimum voltage, maximum voltage and peak-to-peak voltage at the Helper DAC connector. Sketch the waveform.
5. Repeat the scope check at $10$, $100$, $200$ and $300\,\mathrm{Hz}$. Keep the same amplitude setting at each frequency. Choose an amplitude setting that keeps the connector voltage between approximately $-1.2\,\mathrm{V}$ and $+1.2\,\mathrm{V}$ at all five frequencies. Reduce the setting if needed and check the voltages again.

Record the amplitude setting and your oscilloscope readings in your log book. You use them to check the ADC readings in [Task I4](Task_I4_2_2).

````{admonition} Hint: Start the Helper generator
:class: hint dropdown

Inside a Student Pico task, import the modules and use a `with` block. The Helper generates the waveform while the block is running.

```python
import helper
import utime

with helper.function_generator.sine(50, 1.0):
    utime.sleep_ms(30000)
```

Change the frequency or amplitude arguments to repeat the scope check.
````


(Task_I4_2_2)=
#### Task I4: Acquire a frequency sweep

```{admonition} Estimated time: 35 min
:class: estimated-time no-content
```

Keep the Helper DAC setting you checked in [Task I3](Task_I3_2_2). In this task you acquire the generated signal with **ADC0**.

1. Keep ADC0 on DC coupling, 1x input gain and 1x instrumentation amplifier gain. Leave `J302` open and keep `ADC0 U-` connected to `GNDA`.
2. Move the `J301` jumper from `AGND` to `+1.5V`, connecting `IA REF` to `+1.5V`. This shifts the bipolar signal into the ADC range. The $1.5\,\mathrm{V}$ amplifier reference is separate from the Pico ADC's nominal $3.0\,\mathrm{V}$ reference.
3. With the waveform stopped, connect `DAC` on Helper DAC connector `J402` to `ADC0 U+`. Keep the oscilloscope connected so you can compare the signals.
4. Extend your program from [Task I1](Task_I1_2_2) to start a $50\,\mathrm{Hz}$ Helper sine, wait $0.5\,\mathrm{s}$, then collect 1000 readings and their timestamps with a target interval of $0.2\,\mathrm{ms}$. The acquisition must happen while the generator's `with` block is active.
5. Convert the ADC counts to the input signal voltage and plot it against time. With the settings above, the ADC voltage contains an approximately $1.5\,\mathrm{V}$ offset. Subtract that offset to compare the ADC result with the scope voltage at `ADC0 U+`.
6. Record the positive peak voltage and compare it with the scope reading. Check for clipping at the ADC limits.

Then repeat the acquisition for these frequency settings:

- $10\,\mathrm{Hz}$
- $50\,\mathrm{Hz}$
- $100\,\mathrm{Hz}$
- $200\,\mathrm{Hz}$
- $300\,\mathrm{Hz}$

For each frequency, keep the same Helper amplitude setting, wait $0.5\,\mathrm{s}$ for settling, and record 1000 ADC readings and timestamps at a target interval of $0.2\,\mathrm{ms}$. Record the positive peak voltage. Use the oscilloscope to check the source waveform and its frequency.

The target rate is $5000$ samples per second. At $300\,\mathrm{Hz}$, this provides more than ten readings per cycle. The 1000 readings also cover nearly two cycles at $10\,\mathrm{Hz}$. Calculate the actual intervals and average sample rate from your timestamps. Compare the average rate with ten times the frequency measured on the oscilloscope. If it is lower or the timestamps show large gaps relative to one cycle, ask your TA for help before interpreting the peaks.

````{admonition} Hint: Acquire while the Helper runs
:class: hint dropdown

Inside a Student Pico task, put your settling and acquisition loop inside the `with` block. Define `frequency_hz` and the `amplitude_vpp` setting you chose in [Task I3](Task_I3_2_2).

```python
import helper
import utime

with helper.function_generator.sine(frequency_hz, amplitude_vpp):
    utime.sleep_ms(500)
    # Collect the ADC readings and timestamps here.
```

When the block ends, the Helper stops generating the waveform.
````

````{admonition} Hint: Schedule the readings
:class: hint dropdown

Use `ticks_us()` for a target interval of $0.2\,\mathrm{ms}$. Record `start_us` once before the loop. For reading number `i`, wait until its target time, then save the actual timestamp and ADC reading.

```python
target = utime.ticks_add(start_us, i * 200)
while utime.ticks_diff(target, utime.ticks_us()) > 0:
    pass
```
````

**Record in your log book:**

- The measured positive peak voltage at each frequency.
- A plot of measured peak voltage against frequency.
- Whether the peak voltage stays constant across the frequency range.
- Any frequencies with significant deviations and how the ADC peaks compare with the scope.

When you finish, return `J301` to the `AGND` setting used in [Task I1](Task_I1_2_2).


```{admonition} Deep dive: Investigating timing accuracy
:class: deep-dive dropdown

Modify your program from [Task I1](Task_I1_2_2) to analyze timing precision:

- Calculate the time difference between consecutive measurements.
- Find the minimum, maximum, and average time intervals.
- Plot the time intervals over the course of your 100 measurements.

Describe in your log book:
- How consistent the time intervals are
- What fraction of intervals differ from the expected $50\,\mathrm{ms}$
- Possible explanations for any irregularities
- How this timing jitter affects your ability to measure signals

Discuss your result with a TA for your deep dive check-off.
```

```{admonition} Deep dive: Logarithmic frequency range and filter measurement
:class: deep-dive dropdown

In [Task I4](Task_I4_2_2), you used frequency settings of $10$, $50$, $100$, $200$ and $300\,\mathrm{Hz}$. A frequency axis is often logarithmic.

1. Suggest a frequency range with logarithmically spaced values.
2. Measure the $-3\,\mathrm{dB}$ frequency and the slope of the RC filter from [manual 1.2](../week1/1.2_intro_to_oscilloscope_and_filters.ipynb). Take ten measurements per decade.
3. Approximate the frequency characteristic with two lines. Plot the absolute difference between the measured response and this approximation.
4. Determine at which frequency the difference is largest. Explain your result.

Choose a sampling interval that provides at least ten readings per cycle at your highest frequency. Use the oscilloscope to check the input and output voltages. Discuss your result with a TA for your deep dive check-off.
```


### Compare and Conclude

Compare your results with the other pair at your table. Agree on your answers before asking a TA to check your work.


(Task_C1_2_2)=
#### Task C1: Timing accuracy analysis

```{admonition} Estimated time: 10 min
:class: estimated-time no-content
```

```{admonition} Consider Tasks [A1](Task_A1_2_2) and [I1](Task_I1_2_2).
:class: note no-content
```

Complete this table in your log book entry. The duration is measured from the first reading to the last.

| Parameter | Predicted | Measured | Unit |
|:----------|:----------|:---------|:----:|
| First-to-last duration | | | s |
| Sampling interval | 50 | | ms |
| Minimum interval | | | ms |
| Maximum interval | | | ms |
| Mean interval | | | ms |
| Interval range (maximum minus minimum) | | | ms |

How close is the mean interval to the target? Calculate the largest absolute interval error as a percentage of $50\,\mathrm{ms}$. Suggest one cause of the variation and one situation where it would matter.


(Task_C2_2_2)=
#### Task C2: Sampling rate limitations

```{admonition} Estimated time: 15 min
:class: estimated-time no-content
```

```{admonition} Consider Tasks [A2](Task_A2_2_2), [I2](Task_I2_2_2), and [I4](Task_I4_2_2).
:class: note no-content
```

Fill the following table in your notepad:

| Signal Type | Test Frequency [Hz] | Minimum Required Sampling Rate [Hz] | Achieved Sampling Rate [Hz] | Sufficient? |
|:------------|:------------------:|:-----------------------------------:|:---------------------------:|:-----------:|
| Potentiometer (slow) | Not periodic |  |  | Yes/No |
| Sine wave | 50 |  |  | Yes/No |
| Sine wave | 100 |  |  | Yes/No |
| Sine wave | 300 |  |  | Yes/No |

**Questions**:
1. At which frequency do you start to lose the fine details of the sine wave? If none, record that observation.
2. What is the highest signal frequency you measured with confidence using this acquisition setup?
3. How does the measurement interval relate to your maximum sampling rate?
4. How would you modify your program if you wanted to measure frequencies above 500 Hz?

(Task_C3_2_2)=
#### Task C3: Frequency sweep and signal consistency

```{admonition} Estimated time: 15 min
:class: estimated-time no-content
```

```{admonition} Consider Tasks [A4](Task_A4_2_2), [I3](Task_I3_2_2) and [I4](Task_I4_2_2).
:class: note no-content
```

1. Did the measured peak voltage remain constant across the frequency sweep? Compare the ADC peaks with the oscilloscope readings.
2. If variations occurred, which frequencies showed the largest deviations and why?
3. Was the 0.5 s settling time sufficient at each frequency change?
4. How would you optimize the frequency sweep to measure from 50 Hz to 500 Hz more efficiently while maintaining data quality?